# VMC Training — Master Script

End-to-end variational Monte Carlo run: model → Hamiltonian → sampler → optimizer → training → diagnostics.

## Imports

In [1]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt

from qvarnet.train import train
from qvarnet.models.exponential import LogExponentialMLPwithPenalty
from qvarnet.models.deep_set import DeepSet
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian

NOT IMPLEMENTED YET: PairLogExponentialMLPwithGaussianPenalty


## System

In [2]:
N_PARTICLES = 50
DIM         = 1
N_CHAINS    = 5_000
DoF         = N_PARTICLES * DIM
SHAPE       = (N_CHAINS, DoF)

## Model

In [3]:
model = DeepSet(
    phi_hidden_architecture=[2],
    F_hidden_architecture=[2],
    hidden_internal_dimension=2,
    n_particles=N_PARTICLES,
    # kernel_init=jax.nn.initializers.constant(0.0),
    # bias_init=jax.nn.initializers.constant(0.0),
    # kernel_init=jax.nn.initializers.normal(stddev=.01),
    # bias_init=jax.nn.initializers.normal(stddev=.01),
)

model = LogExponentialMLPwithPenalty(
    architecture=[N_PARTICLES, 128, 1],
    hidden_activation=jax.nn.tanh,
    kernel_init=jax.nn.initializers.normal(stddev=.01),
    bias_init=jax.nn.initializers.normal(stddev=.01),
)

IS_LOG_MODEL = True

## Hamiltonian

In [4]:
OMEGA = 1.0

hamiltonian = HarmonicOscillatorHamiltonian(omega=OMEGA)

## Optimizer

In [5]:
LEARNING_RATE = 0.02

optimizer = optax.adam(learning_rate=LEARNING_RATE, eps = 1e-6)

## Sampler

In [6]:
STEP_SIZE            = 0.5
CHAIN_LENGTH         = 100
# THERMALIZATION_STEPS = 10
THINNING_FACTOR      = 1
PBC                  = 40.0

sampler_params = {
    "step_size":            STEP_SIZE,
    "chain_length":         CHAIN_LENGTH + 1,
    "thermalization_steps": CHAIN_LENGTH,
    "thinning_factor":      THINNING_FACTOR,
    "PBC":                  PBC,
}

## Training

In [7]:
import numpy as np

seed = np.random.randint(0, 10000)
N_EPOCHS             = 2_000
RNG_SEED             = seed
WARM_WALKERS         = True
IS_UPDATE_STEP_SIZE  = True
MIN_STEP             = 1e-5
MAX_STEP             = 5.0
SAVE_CHECKPOINTS     = False
CHECKPOINT_PATH      = "./"

history, cm_mean, cm_std = train(
    n_epochs=N_EPOCHS,
    shape=SHAPE,
    model=model,
    optimizer=optimizer,
    sampler_params=sampler_params,
    hamiltonian=hamiltonian,
    rng_seed=RNG_SEED,
    warm_walkers=WARM_WALKERS,
    is_update_step_size=IS_UPDATE_STEP_SIZE,
    is_log_model=IS_LOG_MODEL,
    min_step=MIN_STEP,
    max_step=MAX_STEP,
    save_checkpoints=SAVE_CHECKPOINTS,
    checkpoint_path=CHECKPOINT_PATH,
    use_cm_coords=True,
    n_dim=DIM,
    n_particles=N_PARTICLES,
)

  0%|          | 0/2000 [00:02<?, ?it/s]


XlaRuntimeError: INTERNAL: CUDA error: : CUDA_ERROR_INVALID_VALUE: invalid argument

In [ ]:
print("seed = ", RNG_SEED)

## Extract History

In [ ]:
energies   = np.array([s.energy for s in history]) / N_PARTICLES
stds       = np.array([s.std    for s in history]) / N_PARTICLES
acc_rates  = np.array([float(jnp.mean(s.acceptance_rate)) for s in history])
step_sizes = np.array([float(s.step_size) for s in history])
cm_mean_arr = np.array([float(x) for x in cm_mean])
cm_std_arr  = np.array([float(x) for x in cm_std])

steps   = np.arange(len(history))
E_EXACT = 0.5 * OMEGA  # ground state: 0.5 ℏω (ℏ = 1)

## Summary

In [ ]:
TAIL = 100
final_E   = energies[-TAIL:].mean()
final_std = stds[-TAIL:].mean()
error     = abs(final_E - E_EXACT)

print(f"Exact E₀             : {E_EXACT:.6f}")
print(f"Final E (last {TAIL}) : {final_E:.6f} ± {final_std:.6f}")
print(f"|E - E₀|             : {error:.6f}")
print(f"Relative error       : {error / E_EXACT * 100:.3f}%")
print(f"Final accept rate    : {acc_rates[-TAIL:].mean():.3f}")
print(f"Final step size      : {step_sizes[-1]:.4f}")

## Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# XRANGE = (850, 900)
XRANGE = (200, N_EPOCHS)

ax = axes[0, 0]
ax.plot(steps, energies, lw=0.8, label='E')
ax.fill_between(steps, energies - stds, energies + stds, alpha=0.3, label='±σ')
ax.axhline(E_EXACT, color='red', ls='--', label=f'Exact E₀ = {E_EXACT}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Energy'); ax.set_title('Energy'); ax.legend()
ax.set_xlim(XRANGE)
ax.set_ylim(0.45, 0.6)

ax = axes[0, 1]
ax.semilogy(steps, np.abs(energies - E_EXACT), lw=0.8, color='C1')
ax.set_xlabel('Epoch'); ax.set_ylabel('|E - E₀|'); ax.set_title('Energy error (log scale)')
ax.set_xlim(XRANGE)

ax = axes[0, 2]
ax.semilogy(steps, stds, lw=0.8, color='C2')
ax.set_xlabel('Epoch'); ax.set_ylabel('σ(E)'); ax.set_title('Energy std (log scale)')
ax.set_xlim(XRANGE)

ax = axes[1, 0]
ax.plot(steps, acc_rates, lw=0.8, color='C3')
ax.axhline(0.5, color='red', ls='--', label='target 0.5')
ax.set_xlabel('Epoch'); ax.set_ylabel('Acceptance rate'); ax.set_title('MH acceptance rate'); ax.legend()
ax.set_xlim(XRANGE)

ax = axes[1, 1]
ax.plot(steps, step_sizes, lw=0.8, color='C4')
ax.set_xlabel('Epoch'); ax.set_ylabel('Step size'); ax.set_title('MH step size')
ax.set_xlim(XRANGE)

ax = axes[1, 2]
ax.plot(steps, cm_mean_arr, lw=0.8, color='C5', label='⟨R_cm⟩')
ax.fill_between(steps, cm_mean_arr - cm_std_arr, cm_mean_arr + cm_std_arr, alpha=0.3, label='±std')
ax.axhline(0.0, color='red', ls='--', label='expected 0')
ax.set_xlabel('Epoch'); ax.set_ylabel('R_cm'); ax.set_title('Centre of mass'); ax.legend()
ax.set_xlim(XRANGE)

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Parameter / Gradient Dashboard

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from plot_param_dashboard import compute_stats, save_stats, load_stats, plot_dashboard

param_stats = compute_stats(history, target='params', store_all=True)
grad_stats  = compute_stats(history, target='grads', store_all=True)

In [ ]:
plot_dashboard(param_stats, mode='mean_std', title='Parameter evolution (mean ± std)', xrange=XRANGE);

In [ ]:
plot_dashboard(param_stats, mode='norm', title='Parameter L2 norms', xrange=XRANGE);

In [ ]:
plot_dashboard(param_stats, mode='all', title='all params', xrange=XRANGE);

In [ ]:
plot_dashboard(grad_stats, mode='mean_std', title='Gradient evolution (mean ± std)', xrange=XRANGE);

In [ ]:
plot_dashboard(grad_stats, mode='norm', title='Gradient L2 norms', log_scale=True , xrange=XRANGE);

# Exhaustive exploration of the learning process

## Exhaustive Exploration of the Learning Process

Pick any epoch via `INSPECT_EPOCH` and run the cells below to re-simulate **one full VMC step** using exactly the model parameters stored from that epoch.
Every quantity that shaped that step is computed and visualised:
walkers, local energies, wave function, raw gradients, and the full Adam optimizer state
(first moment μ̂, second moment ν̂, and the actual parameter update Δθ applied).

> **Tip**: to diagnose an instability spike, inspect the epoch *just before* the spike —
> the Adam panel will show which ν̂ values went to zero and which |Δθ| exploded.

In [ ]:
# ── Epoch to inspect ───────────────────────────────────────────────────────
INSPECT_EPOCH  = 1500      # ← change to any value in [0, len(history)-1]

# These MUST match the values used during training
EPS_ADAM       = 1e-6       # Adam ε  (check your optimizer cell)
BETA1, BETA2   = 0.9, 0.999 # Adam β₁, β₂


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from qvarnet.probability import build_prob_fn
from qvarnet.sampling_step import sample_and_process

assert 0 <= INSPECT_EPOCH < len(history), (
    f"INSPECT_EPOCH={INSPECT_EPOCH} out of range — history has {len(history)} entries (0..{len(history)-1})")

# ── 1. State snapshot at the chosen epoch ────────────────────────────────────
state_e    = history[INSPECT_EPOCH]
params_e   = state_e.params
step_e     = float(state_e.step_size)
grads_e    = state_e.grads           # raw ∇_θ L at this epoch
E_stored   = float(state_e.energy)   # total ⟨E⟩ (N particles)
std_stored = float(state_e.std)
acc_stored = float(jnp.mean(state_e.acceptance_rate))

# ── 2. Adam optimizer state & reconstruct the actual Δθ applied ─────────────
# history[i].opt_state = Adam state BEFORE the update at epoch i
# The update used count_inc = count+1 for bias correction
adam_st  = state_e.opt_state[0]   # ScaleByAdamState(count, mu, nu)
t_count  = int(adam_st.count)     # Adam step counter entering this epoch
t1       = t_count + 1            # counter after this epoch's update

mu_prev  = jax.tree_util.tree_leaves(adam_st.mu)
nu_prev  = jax.tree_util.tree_leaves(adam_st.nu)
g_leaves = jax.tree_util.tree_leaves(grads_e)

# Reproduce optax Adam update step
mu_new   = [BETA1*m + (1-BETA1)*g    for m, g in zip(mu_prev, g_leaves)]
nu_new   = [BETA2*v + (1-BETA2)*g**2 for v, g in zip(nu_prev, g_leaves)]
bc1      = 1.0 - BETA1**t1
bc2      = 1.0 - BETA2**t1
mu_hat   = [m / bc1 for m in mu_new]
nu_hat   = [v / bc2 for v in nu_new]
# Actual parameter update: Δθ = -lr * μ̂ / (√ν̂ + ε)   (magnitude shown in plots)
delta_th = [LEARNING_RATE * m / (jnp.sqrt(v) + EPS_ADAM)
            for m, v in zip(mu_hat, nu_hat)]

# ── 3. Human-readable labels for each parameter leaf ────────────────────────
lp_pairs    = jax.tree_util.tree_leaves_with_path(params_e)
leaf_labels = ["/".join(str(k.key) for k in path[1:]) + f"\n{arr.shape}"
               for path, arr in lp_pairs]
n_leaves    = len(leaf_labels)

# Per-leaf aggregate statistics
g_norms  = [float(jnp.linalg.norm(jnp.ravel(g)))   for g in g_leaves]
nu_means = [float(jnp.mean(v))                      for v in nu_hat]
nu_mins  = [float(jnp.min(v))                       for v in nu_hat]
dth_mean = [float(jnp.mean(jnp.abs(d)))             for d in delta_th]
dth_max  = [float(jnp.max(jnp.abs(d)))              for d in delta_th]

# Flat per-parameter arrays (for the individual-param bar chart)
dth_flat   = np.concatenate([np.abs(np.array(d)).ravel() for d in delta_th]) * LEARNING_RATE
nu_flat    = np.concatenate([np.array(v).ravel()         for v in nu_hat])
g_flat     = np.concatenate([np.abs(np.array(g)).ravel() for g in g_leaves])
param_flat = np.concatenate([np.array(p).ravel()         for p in jax.tree_util.tree_leaves(params_e)])
# Label each individual parameter: 'leaf_name[flat_idx]'
param_labels = []
for (path, arr), lbl in zip(lp_pairs, leaf_labels):
    base = lbl.split('\n')[0]
    for idx in range(arr.size):
        param_labels.append(f"{base}[{idx}]" if arr.size > 1 else base)

# ── 4. Re-simulate one MCMC step with params_e ──────────────────────────────
prob_fn_e  = build_prob_fn(model.apply, is_log_model=IS_LOG_MODEL)
key_e      = jax.random.PRNGKey(42)
init_e     = jax.random.normal(key_e, SHAPE) * 0.5

batch_e, _, acc_resim = sample_and_process(
    key=key_e, prob_fn=prob_fn_e, prob_params=params_e,
    init_positions=init_e, step_size=step_e,
    n_chains=N_CHAINS, DoF=DoF,
    n_steps=sampler_params["chain_length"],
    burn_in=sampler_params["thermalization_steps"],
    thinning=sampler_params["thinning_factor"],
    PBC=sampler_params["PBC"],
    is_log_prob=IS_LOG_MODEL,
)

# ── 5. Local energies ────────────────────────────────────────────────────────
e_loc_np  = np.array(hamiltonian.local_energy(
    params_e, batch_e, model.apply, is_log_model=IS_LOG_MODEL))
E_resim   = float(np.mean(e_loc_np))
std_resim = float(np.std(e_loc_np))
SEM_resim = std_resim / np.sqrt(len(e_loc_np))

# ── 6. Single-particle wave function  (vary x₀, all others at 0) ────────────
x_probe    = np.linspace(-7.5, 7.5, 500)
X_1d       = np.zeros((500, DoF), dtype=np.float32)
X_1d[:, 0] = x_probe
log_psi_1d = np.array(model.apply(params_e, jnp.array(X_1d))).squeeze()
psi_sq_1d  = np.exp(2.0 * log_psi_1d)
psi_sq_1d /= psi_sq_1d.max()
# Exact reference for 1-particle HO ground state (N-1 others fixed at 0)
log_psi_exact = -0.5 * x_probe**2  # up to additive constant
log_psi_exact -= log_psi_exact.max()
log_psi_exact += log_psi_1d.max()  # align vertical offset for comparison
psi_sq_exact   = np.exp(-x_probe**2); psi_sq_exact /= psi_sq_exact.max()

# ── 7. Summary print ────────────────────────────────────────────────────────
sep = '=' * 64
print(sep)
print(f"  EPOCH {INSPECT_EPOCH} / {len(history)-1}     Adam step counter: {t_count} → {t1}")
print(sep)
print(f"  Stored  ⟨E⟩/N = {E_stored/N_PARTICLES:.8f}   σ/N = {std_stored/N_PARTICLES:.2e}")
print(f"  Re-sim  ⟨E⟩/N = {E_resim/N_PARTICLES:.8f}   σ/N = {std_resim/N_PARTICLES:.2e}")
print(f"  SEM/N          = {SEM_resim/N_PARTICLES:.2e}")
print(f"  step_size      = {step_e:.6f}    acceptance = {acc_stored:.4f}")
print(f"  bias-corr bc1  = {bc1:.8f}    bc2 = {bc2:.8f}")
print(f"  max |Δθ|       = {max(dth_max)*LEARNING_RATE:.4e}  (leaf: "
      f"{leaf_labels[np.argmax(dth_max)].split(chr(10))[0]})")
print(f"  max |eff_step| = {max(dth_max):.4e}  = |μ̂|/(√ν̂+ε) before ×lr")
pct = np.percentile(e_loc_np, [0, 1, 25, 50, 75, 99, 100])
print(f"  E_loc [min 1% 25% 50% 75% 99% max] = "
      f"{pct[0]:.2f}  {pct[1]:.2f}  {pct[2]:.2f}  {pct[3]:.2f}  {pct[4]:.2f}  {pct[5]:.2f}  {pct[6]:.2f}")
print(sep)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ep_range   = np.arange(len(history))
idx        = np.arange(n_leaves)
all_idx    = np.arange(len(param_labels))

# Colour-code leaves for the individual-param chart
leaf_sizes = [arr.size for _, arr in lp_pairs]
leaf_colors = plt.cm.tab10(np.linspace(0, 1, n_leaves))
param_colors = np.concatenate([[leaf_colors[i]]*s for i, s in enumerate(leaf_sizes)])

fig = plt.figure(figsize=(22, 15))
fig.suptitle(
    f"Epoch {INSPECT_EPOCH} — Full VMC Step Diagnostic"
    f"    (⟨E⟩/N={E_resim/N_PARTICLES:.5f}  σ/N={std_resim/N_PARTICLES:.2e}  step={step_e:.4f})",
    fontsize=13, fontweight="bold")
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.52, wspace=0.35)

# ── Row 0: training-curve context ───────────────────────────────────────────
def mark_epoch(ax):
    ax.axvline(INSPECT_EPOCH, color="black", lw=1.2, ls="--", alpha=0.7)
    ax.scatter([INSPECT_EPOCH], [energies[INSPECT_EPOCH]], s=50, zorder=6,
               color="black") if ax.get_title().startswith("Energy") else None

ax = fig.add_subplot(gs[0, 0])
ax.plot(ep_range, energies, lw=0.5, color="C0")
ax.fill_between(ep_range, energies-stds, energies+stds, alpha=0.2, color="C0")
ax.axhline(E_EXACT, color="red", ls="--", lw=0.9, label=f"E₀={E_EXACT}")
ax.axvline(INSPECT_EPOCH, color="black", lw=1.2, ls="--", alpha=0.7)
ax.scatter([INSPECT_EPOCH], [energies[INSPECT_EPOCH]], s=60, zorder=6, color="black")
ax.set_xlabel("Epoch"); ax.set_ylabel("E/N"); ax.set_title("Energy")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[0, 1])
ax.semilogy(ep_range, stds, lw=0.5, color="C2")
ax.axvline(INSPECT_EPOCH, color="black", lw=1.2, ls="--", alpha=0.7)
ax.scatter([INSPECT_EPOCH], [stds[INSPECT_EPOCH]], s=60, zorder=6, color="black")
ax.set_xlabel("Epoch"); ax.set_ylabel("σ(E)/N"); ax.set_title("Energy std (log)")

ax = fig.add_subplot(gs[0, 2])
ax.plot(ep_range, acc_rates, lw=0.5, color="C3")
ax.axhline(0.5, color="red", ls="--", lw=0.9)
ax.axvline(INSPECT_EPOCH, color="black", lw=1.2, ls="--", alpha=0.7)
ax.set_ylim(0, 1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Acceptance"); ax.set_title("MH acceptance rate")

ax = fig.add_subplot(gs[0, 3])
ax.plot(ep_range, step_sizes, lw=0.5, color="C4")
ax.axvline(INSPECT_EPOCH, color="black", lw=1.2, ls="--", alpha=0.7)
ax.scatter([INSPECT_EPOCH], [step_sizes[INSPECT_EPOCH]], s=60, zorder=6, color="black")
ax.set_xlabel("Epoch"); ax.set_ylabel("Step size"); ax.set_title("MH step size")

# ── Row 1: sampling quality at this epoch ───────────────────────────────────
batch_np = np.array(batch_e)

ax = fig.add_subplot(gs[1, 0])
ax.hist(batch_np.ravel(), bins=80, density=True, color="C0", alpha=0.7, label="walkers")
xx = np.linspace(-7.5, 7.5, 400)
ax.plot(xx, np.exp(-0.5*xx**2)/np.sqrt(2*np.pi), "r--", lw=1.3, label="N(0,1)")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.set_title(f"Walker positions (all {DoF} coords × {len(batch_np)} walkers)")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[1, 1])
lo, hi = np.percentile(e_loc_np, [1, 99])
ax.hist(e_loc_np, bins=80, density=True, color="C1", alpha=0.75)
ax.axvline(E_resim,           color="black", lw=1.3, label=f"⟨E⟩={E_resim:.2f}")
ax.axvline(E_EXACT*N_PARTICLES, color="red",   lw=1.2, ls="--",
           label=f"exact={E_EXACT*N_PARTICLES:.2f}")
ax.set_xlabel("$E_{loc}$ (total, clipped 1–99%)"); ax.set_ylabel("density")
ax.set_title(f"Local energy dist\nmin={e_loc_np.min():.1f}  max={e_loc_np.max():.1f}  "
             f"σ={std_resim:.4f}")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[1, 2])
ax.plot(x_probe, psi_sq_1d,   color="C5",  lw=1.8, label="model")
ax.plot(x_probe, psi_sq_exact, color="red", lw=1.0, ls="--", label="exact $e^{-x^2}$")
ax.set_xlabel("x"); ax.set_ylabel("|ψ(x,0,…)|² (norm. to 1)")
ax.set_title("Single-particle wave function")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[1, 3])
ax.plot(x_probe, log_psi_1d,    color="C6",  lw=1.8, label="model")
ax.plot(x_probe, log_psi_exact, color="red", lw=1.0, ls="--", label="exact $-x^2/2$+c")
ax.set_xlabel("x"); ax.set_ylabel("log|ψ(x,0,…)|")
ax.set_title("log|ψ| single particle")
ax.legend(fontsize=7)

# ── Row 2: Adam optimizer state ─────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
bars = ax.bar(idx, g_norms, color=[leaf_colors[i] for i in range(n_leaves)], alpha=0.85)
ax.set_xticks(idx)
ax.set_xticklabels([l.split('\n')[0] for l in leaf_labels],
                   rotation=40, ha="right", fontsize=6.5)
ax.set_ylabel("‖∇θ‖₂"); ax.set_title("Gradient L2 norm per leaf")

ax = fig.add_subplot(gs[2, 1])
ax.bar(idx, nu_means, color=[leaf_colors[i] for i in range(n_leaves)], alpha=0.85,
       label="mean ν̂")
ax.bar(idx, nu_mins,  color=[leaf_colors[i] for i in range(n_leaves)], alpha=0.4,
       label="min ν̂")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels([l.split('\n')[0] for l in leaf_labels],
                   rotation=40, ha="right", fontsize=6.5)
ax.set_ylabel("ν̂  (Adam 2nd moment)")
ax.set_title("Adam ν̂ per leaf  (log scale)\nsmall → large eff. step → instability risk")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[2, 2])
ax.bar(idx, dth_mean, color=[leaf_colors[i] for i in range(n_leaves)], alpha=0.85,
       label="mean |Δθ|")
ax.bar(idx, dth_max,  color=[leaf_colors[i] for i in range(n_leaves)], alpha=0.3,
       label="max |Δθ|")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels([l.split('\n')[0] for l in leaf_labels],
                   rotation=40, ha="right", fontsize=6.5)
ax.set_ylabel("η · |μ̂| / (√ν̂ + ε)  =  |Δθ|")
ax.set_title("Actual parameter update |Δθ| per leaf\n(mean + max, log scale)")
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[2, 3])
bars = ax.bar(all_idx, dth_flat, color=param_colors, alpha=0.85)
ax.set_yscale("log")
ax.set_xticks(all_idx)
ax.set_xticklabels(param_labels, rotation=45, ha="right", fontsize=5.5)
ax.set_ylabel("η · |μ̂| / (√ν̂ + ε)")
ax.set_title("Actual |Δθ| per individual parameter\n(colors = leaves)")
# Add a horizontal reference line at a 'normal' update magnitude
ax.axhline(LEARNING_RATE * 0.1, color="grey", ls="--", lw=0.8, label="0.1×lr ref")
ax.legend(fontsize=7)

fname = f"epoch_{INSPECT_EPOCH:06d}_diagnostic.png"
plt.savefig(fname, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved → {fname}")
